# Member 2 — Spatial Hotspot Analysis
Using the uploaded water-quality dataset.

In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.append(str(Path("..").resolve()))
from src.spatial.geo_utils import prepare_locations
from src.spatial.hotspot_classifier import numeric_item_values, build_water_quality_matrix, classify_risk
from src.spatial.mapping import make_hotspot_map
from src.temporal.time_grouping import add_time_groups
from src.temporal.trend_tracking import site_trends, classify_hotspot_evolution

df = pd.read_csv("../data/raw/water_quality.csv")
df.shape, df.columns.tolist()

/Users/geetikarupani/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


((100, 16),
 ['siteid',
  'siteengname',
  'countyen',
  'townshipen',
  'basinen',
  'riveren',
  'twd97lon',
  'twd97lat',
  'twd97tm2x',
  'twd97tm2y',
  'sampledate',
  'itemengname',
  'itemengabbreviation',
  'itemvalue',
  'itemunit',
  'noteen'])

In [2]:
locations = prepare_locations(df)
locations

,siteid,siteengname,countyen,townshipen,basinen,riveren,twd97lat,twd97lon
0,1724,Waizihwai Bridge,Yilan County,Sanxing Township,Lanyang River,Luodong River,24.702833,121.753083
33,1686,Chengde Bridge,Pingtung County,Wanluan Township,Donggang River,Donggang River,22.613611,120.588694
66,1685,Yungan Bridge,Tainan City,Annan District,Yanshuei River,Yanshuei River,23.051389,120.239500
99,1684,Dungshrnan Bridge,Chiayi County,Dongshih Township,Puzih River,Puzih River,23.443694,120.168861


In [3]:
matrix = build_water_quality_matrix(df)
print(matrix.shape)
matrix.head()

(2, 30)


itemengname,siteid,siteengname,sampledate,twd97lat,twd97lon,Arsenic,Biochemical Oxygen Demand,Cadmium,Chemical Oxygen Demand,Chloride,...,Nickel,Nitrate-Nitrogen,River Pollution Index,Silver,Suspended Solid,Temperature,Total-Phosphate,Water Temperature,Zinc,pH
0,1685,Yungan Bridge,2026-07-16 17:12:46,23.051389,120.239500,0.0078,2.5,0.001,23.8,460.0,...,0.008,3.60,4.5,0.001,18.1,29.8,12.10,30.0,0.016,7.41
1,1724,Waizihwai Bridge,2026-07-03 09:47:04,24.702833,121.753083,0.0029,6.8,0.001,21.8,NaN,...,0.005,0.57,4.5,0.001,292.0,29.2,0.23,27.6,0.026,7.87


In [4]:
risk = classify_risk(matrix)
risk[["siteengname","sampledate","risk_score","risk_level","risk_basis"]]

itemengname,siteengname,sampledate,risk_score,risk_level,risk_basis
0,Yungan Bridge,2026-07-16 17:12:46,4.5,LOW,River Pollution Index
1,Waizihwai Bridge,2026-07-03 09:47:04,4.5,LOW,River Pollution Index


In [5]:
risk = add_time_groups(risk)
risk[["siteengname","sampledate","year","month","season","risk_level"]]

itemengname,siteengname,sampledate,year,month,season,risk_level
0,Yungan Bridge,2026-07-16 17:12:46,2026,7,Summer,LOW
1,Waizihwai Bridge,2026-07-03 09:47:04,2026,7,Summer,LOW


In [6]:
evolution = classify_hotspot_evolution(risk)
evolution

,siteengname,observations,first_risk,latest_risk,mean_score,hotspot_status
0,Waizihwai Bridge,1,LOW,LOW,4.5,SINGLE_OBSERVATION
1,Yungan Bridge,1,LOW,LOW,4.5,SINGLE_OBSERVATION


In [7]:
map_df = risk.dropna(subset=["twd97lat","twd97lon"]).copy()
m = make_hotspot_map(map_df, "../hotspot_map.html")
m